# 🤖 Multi-Model Training, Evaluation, & Architecture Selection

## 📦 Step 1: Modeling Environment Setup & Dependency Ingestion

In this cell, we initialize our modeling ecosystem by importing the specialized libraries required to execute cross-validated multi-model training, hyperparameter optimization, and comprehensive medical evaluation metrics.

* **Core Data Engine:** `pandas`, `numpy`
* **Linear & Distance-Based Candidates:** `LogisticRegression`, `KNeighborsClassifier`, `SVC` (Support Vector Machine)
* **Probabilistic & Tree-Based Candidates:** `GaussianNB` (Naive Bayes), `DecisionTreeClassifier`, `RandomForestClassifier`, `XGBClassifier`
* **Robust Validation & Hyperparameter Tuning:** `GridSearchCV`, `StratifiedKFold`, `cross_val_score`
* **Clinical Performance Metrics:** `accuracy_score`, `precision_score`, `recall_score`, `f1_score`, `roc_auc_score`
* **Production Serialization & Utilities:** `joblib`, `logging`, `pathlib`, `os`, `sys`

In [43]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.svm import SVC
from pathlib import Path
import pandas as pd 
import numpy as np 
import importlib 
import logging 
import joblib
import os
import sys 

## 📝 Step 2: Production-Grade Logging Pipeline Initialization

This phase establishes a robust, dual-destination logging infrastructure. By setting up centralized logging before model training begins, we ensure complete traceability of model parameters, execution times, and evaluation results.

### ⚙️ Logging Architecture Breakdown
We construct two distinct handlers to cleanly separate high-level development feedback from deep execution debugging:

* **Persistent File Handler (`DEBUG` Level):** Creates a dedicated `logs/` directory at the project root and writes highly detailed tracking logs (`heart_disease_pipeline.log`). It records precise script execution paths, timestamps, file names, line numbers, and granular debug data.
* **Notebook Console Handler (`INFO` Level):** Streams clean, distraction-free progress tracking messages with clear status updates directly below your active notebook cells, hiding the underlying debug noise.

In [ ]:
#crates logs folder first
BASE_DIR = Path(os.getcwd())
PROJECT_ROOT = BASE_DIR.parent
LOGS_DIR = PROJECT_ROOT / 'logs'
LOGS_DIR.mkdir(parents=True, exist_ok=True)

#reset logging configuration
importlib.reload(logging)
#define logging path 
log_filename = LOGS_DIR / "heart_disease_pipeline.log"
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)  # Capture everything from DEBUG up to CRITICAL

if logger.hasHandlers():
    logger.handlers.clear()

# File Handler: Writes raw, highly detailed metrics into your persistent log file
file_handler = logging.FileHandler(log_filename, mode='w')
file_handler.setLevel(logging.DEBUG)
file_formatter = logging.Formatter('%(asctime)s | %(levelname)-8s | %(filename)s (Line:%(lineno)d) | %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

# Console Handler: Streams clean updates with emojis directly underneath your notebook cell
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)  # Keep screen output clean (ignore debug noise)
console_formatter = logging.Formatter('[%(levelname)s] %(message)s')
console_handler.setFormatter(console_formatter)
logger.addHandler(console_handler)

logging.info("INITIALIZING ARCHITECTURAL MODULE: MULTI-MODEL TRAINING PIPELINE")

[INFO] INITIALIZING ARCHITECTURAL MODULE: MULTI-MODEL TRAINING PIPELINE


## ⚙️ Step 3: Multi-Model Blueprint Definition & Evaluation Pipeline Engine

This phase establishes our comparative modeling suite and constructs an automated execution engine to train, test, and evaluate each candidate model uniformly.

### 📋 Architectural Blueprint & Execution Logic

* **Diverse Candidate Spectrum:** We define a comprehensive dictionary mapping 7 core machine learning architectures. This array spans across linear models, distance-based neighborhood models, probabilistic networks, and high-performance tree ensembles.
* **Automated Evaluation Loop:** The `train_and_evaluate_pipline` function standardizes our operational testing framework. It systematically tracks metrics across each model using our balanced validation split.
* **Fault-Tolerant Training Mechanics:** Embedded within a robust `try-except` defensive architecture, any localized convergence or optimization crash within a specific algorithm will be securely caught and logged as an error, protecting the pipeline from structural halts.

In [45]:

models_blueprint = {
    'Logistic_Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive_Bayes': GaussianNB(),
    'Decision_Tree': DecisionTreeClassifier(random_state=42),
    'Random_Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss') 
}

logging.info(f"Successfully loaded blueprint dictionary with {len(models_blueprint)} models ready for training.")

def train_and_evaluate_pipline(models : dict, x_train,x_test,y_train,y_test):
    performance_matrix = {}
    logging.info(f"Dimenstions verified : x_train {x_train.shape} | x_test {x_test.shape}")
    logging.info(f"Startig multimodel Excution loop")
    for model_name,model_obj in models.items():
        logging.info(f"processing model {model_name}")
        try:
            model_obj.fit(x_train,y_train)
            y_pred = model_obj.predict(x_test)
            y_prob = model_obj.predict_proba(x_test)[:,1] if hasattr(model_obj,"predict_proba" ) else None
            metrics = {
                'Accuracy':accuracy_score(y_test,y_pred),
                "Precision":precision_score(y_test,y_pred,zero_division=0),
                "Recall":recall_score(y_test,y_pred,zero_division=0),
                "F1-Score":f1_score(y_test,y_pred,zero_division=0),
                "ROC_AUC":roc_auc_score(y_test,y_prob) if y_prob is not None else "N/A"             
            }
            performance_matrix[model_name] = metrics
            logging.info(f"[{model_name}] Evaluation complete. Recall: {metrics['Recall']:.4f} | ROC-AUC: {f'{metrics['ROC_AUC']:.4f}' if isinstance(metrics['ROC_AUC'], float) else 'N/A'}")

        except Exception as e:
            logging.error(f"Execution crash model {model_name}. Error details: {str(e)}")

    logging.info("Multi-model excution complete")
    return performance_matrix



[INFO] Successfully loaded blueprint dictionary with 7 models ready for training.


## 🚀 Step 4: Data Loading, Pipeline Execution, & Baseline Evaluation

This phase connects our data pipeline to our modeling engine. We ingest the decoupled, scaled data structures from storage, execute our 7-model training loop, and rank the initial results.

### 📋 Structural Code Mechanics

* **Dimensional Alignment via Vector Flattening (`.values.ravel()`):** When loading target variables from CSV files, pandas automatically interprets them as a 2D data frame structure. Because Scikit-Learn classification algorithms strictly expect a clean, 1D array for labels, we chain `.values.ravel()` to extract the raw underlying NumPy array and flatten it. This completely prevents shape misalignment warnings.
* **Clinical Optimization Hierarchy:** After converting the raw performance dictionary into a clean evaluation dataframe, we explicitly sort the results by **Recall** in descending order. In a medical diagnostic framework, maximizing Recall is paramount because it minimizes False Negatives—ensuring that high-risk cardiac patients are not mistakenly classified as healthy.

In [46]:
x_train =pd.read_csv("../Data/Final/x_train_scaled.csv")
x_test = pd.read_csv("../Data/Final/x_test_scaled.csv")
#pandas read  csv as dataframe with shape (n_samples,n_features) but we need to be 1D array 
y_train = pd.read_csv("../Data/Final/y_train.csv").values.ravel()
y_test = pd.read_csv("../Data/Final/y_test.csv").values.ravel()

raw_baseline_results = train_and_evaluate_pipline(models_blueprint,x_train,x_test,y_train,y_test)
#trasnform the resulst into a datafram 
df_baseline = pd.DataFrame(raw_baseline_results).T

# Sort by Recall primarily
df_baseline = df_baseline.sort_values(by='Recall', ascending=False)
print(df_baseline)

[INFO] Dimenstions verified : x_train (241, 18) | x_test (61, 18)
[INFO] Startig multimodel Excution loop
[INFO] processing model Logistic_Regression
[INFO] [Logistic_Regression] Evaluation complete. Recall: 0.8485 | ROC-AUC: 0.8690
[INFO] processing model SVM
[INFO] [SVM] Evaluation complete. Recall: 0.9394 | ROC-AUC: 0.9026
[INFO] processing model KNN
[INFO] [KNN] Evaluation complete. Recall: 0.9091 | ROC-AUC: 0.8810
[INFO] processing model Naive_Bayes
[INFO] [Naive_Bayes] Evaluation complete. Recall: 0.3636 | ROC-AUC: 0.7825
[INFO] processing model Decision_Tree
[INFO] [Decision_Tree] Evaluation complete. Recall: 0.7576 | ROC-AUC: 0.7181
[INFO] processing model Random_Forest
[INFO] [Random_Forest] Evaluation complete. Recall: 0.8788 | ROC-AUC: 0.8934
[INFO] processing model XGBoost
[INFO] [XGBoost] Evaluation complete. Recall: 0.8485 | ROC-AUC: 0.8593
[INFO] Multi-model excution complete
                     Accuracy  Precision    Recall  F1-Score   ROC_AUC
SVM                  0.83

## 🎯 Step 5: Hyperparameter Tuning Strategy & Cross-Validation Wrapper Setup

This phase upgrades our modeling suite by wrapping our complex, non-linear algorithms in hyperparameter optimization grids while maintaining strict validation standards.

### 📋 Strategy Breakdown: Tuning vs. Baselines

We intentionally left out KNN, Logistic Regression, and Naïve Bayes from `GridSearchCV` for two reasons: architectural simplicity and mathematical nature.

* **Naïve Bayes has almost nothing to tune:** Algorithms like `GaussianNB` don't have complex structural settings like tree depth or regularizations. They just calculate simple probabilities directly from your dataset features. There is no "search grid" that would drastically alter its behavior.
* **Baselines must remain baselines:** In a professional machine learning pipeline, you need a raw, un-tuned model to serve as a benchmark. If you tune every single thing immediately, you won't know if a complex model like Random Forest is actually better, or if you just needed a simple, well-tuned linear model.
* **KNN & Logistic Regression are already solid:** Your `KNeighborsClassifier(n_neighbors=5)` and standard `LogisticRegression` already yielded extremely strong results (0.9091 and 0.8485 Recall respectively). That being said, if you wanted to tune them later, you absolutely could wrap them in `GridSearchCV` exactly like the others. For example, you could test different `n_neighbors` values ($3, 5, 7, 11$) for KNN or different regularization strengths (`C`) for Logistic Regression.

### 🛡️ Cross-Validation and Metric Evaluation Strategy

#### **We should validate all the models same way**
To ensure absolute fairness, every model—whether it's an un-tuned baseline or wrapped in a complex hyperparameter search—must be tested against the exact same validation strategy. 

We use a 5-fold **StratifiedKFold** (`cv_strategy`) inside our grids to preserve the target class balance across every split. For our search optimization target, we select `scoring='roc_auc'`, focusing the parameter tuning on optimizing the models' overall capability to separate healthy patients from high-risk cardiac patients across all threshold variants.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svm_param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.01, 0.1]
}

rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 4, 6, 10],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', None] #even the classes are balanced so None is better but we take all in consideration
}

xgb_param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 6]
}

updated_blueprint = {
    'Logistic_Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive_Bayes': GaussianNB(),
    'Decision_Tree': DecisionTreeClassifier(random_state=42),
    'SVM': GridSearchCV(SVC(probability=True, random_state=42), param_grid=svm_param_grid, cv=cv_strategy, scoring='roc_auc', n_jobs=-1),
    'Random_Forest': GridSearchCV(RandomForestClassifier(random_state=42), param_grid=rf_param_grid, cv=cv_strategy, scoring='roc_auc', n_jobs=-1),
    'XGBoost': GridSearchCV(XGBClassifier(random_state=42, eval_metric='logloss'), param_grid=xgb_param_grid, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
}

logging.info(f"Tuning blueprint registered with GridSearch wrappers for SVM, RF, and XGBoost.")

[INFO] Tuning blueprint registered with GridSearch wrappers for SVM, RF, and XGBoost.


## ⚙️ Step 6: Unified Cross-Validation, Training, & Serialization Engine

This phase upgrades our execution engine to a production-ready training and artifact-saving utility. The `train_evaluate_and_save_pipeline` function automates cross-validation, hyperparameter selection, testing, and model persistence all in a single run.

### 📋 Architectural Code Mechanics

* **Forced Cross-Validation Benchmarking (`cross_val_score`):** To enforce absolute validation fairness across all 7 candidates, we pass every single model through `cross_val_score` using our predefined `cv_strategy`. We explicitly target `scoring='recall'` to see how stably each model catches clinical heart disease cases during internal cross-validation folds.
* **Automated GridSearch Extraction (`best_params_`):** The engine uses reflection (`hasattr`) to see if a model is wrapped in a `GridSearchCV` container. If it is, the code extracts the optimal hyperparameters, logs them, and sets the fine-tuned `.best_estimator_` as the active model for serialization and final testing.
* **Production Serialization (`joblib.dump`):** Once an active model is completely trained and resolved, the function automatically serializes and saves it as a `.joblib` binary inside our dedicated `models/` folder. This ensures that every single trained architecture is permanently archived and instantly ready for a backend deployment.

In [48]:

def train_evaluate_and_save_pipeline(models : dict, x_train,x_test,y_train,y_test,cv_strategy,project_root_path):
    performance_matrix = {}
    models_dir = project_root_path / 'models'
    models_dir.mkdir(parents=True, exist_ok=True)
    logging.info(f"Dimenstions verified : x_train {x_train.shape} | x_test {x_test.shape}")
    logging.info(f"Starting production cross-validation, training, and serialization loop...")
    for model_name,model_obj in models.items():
        logging.info(f"processing model {model_name}")
        try:
            model_obj.fit(x_train,y_train)
            #we need to force cross validation on all models 
            cv_scores = cross_val_score(model_obj, x_train, y_train, cv=cv_strategy, scoring='recall', n_jobs=-1)
            mean_cv_recall = cv_scores.mean()
            #check if it's gridseearhc object 
            if hasattr(model_obj, 'best_params_'):
                logging.info(f"[{model_name}] Optimal Hyperparameters Found: {model_obj.best_params_}")
                active_model = model_obj.best_estimator_
            else:
                active_model = model_obj
            model_save_path = models_dir / f"{model_name}_model.joblib"
            joblib.dump(active_model, model_save_path)
            logging.info(f"[{model_name}] Saved successfully to: {model_save_path.name}")

            y_pred = active_model.predict(x_test)
            y_prob = active_model.predict_proba(x_test)[:,1] if hasattr(active_model,"predict_proba" ) else None
            metrics = {
                'Accuracy':accuracy_score(y_test,y_pred),
                "Precision":precision_score(y_test,y_pred,zero_division=0),
                "Recall":recall_score(y_test,y_pred,zero_division=0),
                "F1-Score":f1_score(y_test,y_pred,zero_division=0),
                "ROC_AUC":roc_auc_score(y_test,y_prob) if y_prob is not None else "N/A"             
             }
            performance_matrix[model_name] = metrics
            logging.info(f"[{model_name}] Evaluation complete. Recall: {metrics['Recall']:.4f} | ROC-AUC: {f'{metrics['ROC_AUC']:.4f}' if isinstance(metrics['ROC_AUC'], float) else 'N/A'}")

        except Exception as e:
            logging.error(f"Execution crash model {model_name}. Error details: {str(e)}")

    logging.info("Multi-model excution complete")
    return performance_matrix

In [50]:
PROJECT_ROOT = Path.cwd().parent
final_production_results = train_evaluate_and_save_pipeline(
    models=updated_blueprint,
    x_train=x_train,
    x_test=x_test,
    y_train=y_train,
    y_test=y_test,
    cv_strategy=cv_strategy,
    project_root_path=PROJECT_ROOT
)

# Convert results into a clean dataframe
df_production_matrix = pd.DataFrame(final_production_results).T
df_production_matrix = df_production_matrix.sort_values(by='Recall', ascending=False)
df_production_matrix

[INFO] Dimenstions verified : x_train (241, 18) | x_test (61, 18)
[INFO] Starting production cross-validation, training, and serialization loop...
[INFO] processing model Logistic_Regression
[INFO] [Logistic_Regression] Saved successfully to: Logistic_Regression_model.joblib
[INFO] [Logistic_Regression] Evaluation complete. Recall: 0.8485 | ROC-AUC: 0.8690
[INFO] processing model KNN
[INFO] [KNN] Saved successfully to: KNN_model.joblib
[INFO] [KNN] Evaluation complete. Recall: 0.9091 | ROC-AUC: 0.8810
[INFO] processing model Naive_Bayes
[INFO] [Naive_Bayes] Saved successfully to: Naive_Bayes_model.joblib
[INFO] [Naive_Bayes] Evaluation complete. Recall: 0.3636 | ROC-AUC: 0.7825
[INFO] processing model Decision_Tree
[INFO] [Decision_Tree] Saved successfully to: Decision_Tree_model.joblib
[INFO] [Decision_Tree] Evaluation complete. Recall: 0.7576 | ROC-AUC: 0.7181
[INFO] processing model SVM
[INFO] [SVM] Optimal Hyperparameters Found: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
[INF

,Accuracy,Precision,Recall,F1-Score,ROC_AUC
Random_Forest,0.836066,0.794872,0.939394,0.861111,0.906926
KNN,0.819672,0.789474,0.909091,0.845070,0.880952
SVM,0.803279,0.783784,0.878788,0.828571,0.880952
XGBoost,0.786885,0.763158,0.878788,0.816901,0.874459
Logistic_Regression,0.819672,0.823529,0.848485,0.835821,0.869048
Decision_Tree,0.721311,0.735294,0.757576,0.746269,0.718074
Naive_Bayes,0.622951,0.857143,0.363636,0.510638,0.782468
